# Projeto Final: análise das candidaturas da eleição de 2022

**Autores:** Felipe, Denise e Fabrício

## Datasets utilizados
- **Base principal:** consulta_cand_2022_BRASIL.csv
- **Base auxiliar:**  consulta_cand_complementar_2022_BRASIL.csv

## Perguntas que serão respondidas:
1. Cargos com mais candidaturas
2. Distribuição por gênero
3. Distribuição por raça/cor
4. Taxa de eleição por escolaridade
5. Candidaturas e eleitos por UF
6. Taxa de eleição por gênero
7. Resultado por faixa etária

## 1. Importando as bibliotecas Pandas e Numpy

In [1]:
# Para facilitar no desenvolvimento, "apelidamos" as bibliotecas no momento da importação
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 2. Criando o DataFrame

### 2.1. Erro na primeira tentativa de criação do DataFrame

In [ ]:
"""
caminho note pessoal
pasta_arquivo = "/Users/felipealbanez/Github/CAIXAVERSO_DFF/Projeto_Final_TSE_2022/consulta_cand_2022_BRASIL.csv"

caminho Trabalho
pasta_arquivo = "/Users/cxxxxxx/OneDrive - Caixa Economica Federal/Área de Trabalho/CAIXA Verso/Projeto Final/Base de dados/consulta_cand_2022_BRASIL.csv"

df_original = pd.read_csv(pasta_arquivo)
df_original.head()

erro gerado:
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc7 in position 838: invalid continuation byte

"""

### 2.2. Segunda tentativa - modo "a brasileira"

In [79]:
""""
sep=';': Os dados usam ponto e vírgula para separar as colunas
decimal=",": números decimais usam vírgula
encoding='latin-1': Evita erros de leitura com acentos e caracteres da língua portuguesa.
"""
# substitua o caminho abaixo pela pasta onde está salvo a base de dados .csv
# não esquecer que o nome do arquivo e a extensão deve estar inclusos

# caminho note pessoal
pasta_arquivo = "../dados/consulta_cand_2022_BRASIL.csv"

# caminho Trabalho
# pasta_arquivo = "/Users/cxxxxxx/OneDrive - Caixa Economica Federal/Área de Trabalho/CAIXA Verso/Projeto Final/Base de dados/consulta_cand_2022_BRASIL.csv"

# cria o dataframe lendo o arquivo .csv
df_original = pd.read_csv(pasta_arquivo, sep=';', decimal=',',encoding='latin-1')

In [ ]:
# vizualisa as 5 primeiras linhas do dataframe
df_original.head()

## 3. Conhecendo o DataFrame

### 3.1. Dimensões e Estrutura do DataFrame

In [ ]:
# shape: informa o número de linhas e colunas
print(f'Shape: {df_original.shape}')
print(f'Quantidade de linhas: {df_original.shape[0]}')
print(f'Quantidade de colunas: {df_original.shape[1]}')

In [ ]:
# info: informa os tipos, faltantes, memória
df_original.info()        

In [ ]:
# describle aula
display(df_original.describe().round(2))
display(df_original.describe(include=['object', 'string']))

### 3.2 Avaliando as categorizaçãos

In [ ]:
# Mostra os nomes das colunas.
df_original.columns

In [ ]:
# Avaliação de algumas colunas que serão utilizadas
print(df_original['DS_GENERO'].value_counts(dropna=False))
print()
print(df_original['DS_COR_RACA'].value_counts(dropna=False))
print()
print(df_original['DS_GRAU_INSTRUCAO'].value_counts(dropna=False))

## 4. Tratamento de Faltantes (NaN)

### 4.1. Faltantes puros

In [ ]:
# cria um df chamado Faltantes
# são criadas tres colunas para o df 
# isna.sum: percorre a coluna e retorna a soma dos faltantes
# isna.mean: percorre a coluna e retorna a média dos faltantes em relação ao número total de itens na coluna
# nunique: conta quantos valores diferentes cada coluna tem
faltantes = pd.DataFrame({
    'Faltantes': df_original.isna().sum(),
    '%': (df_original.isna().mean() * 100).round(2),
    'distintos': df_original.nunique(),
})

# filta somente as colunas que não possui faltantes
# sort e ascending: ordena pela coluna faltante do maior para o menor
faltantes[faltantes['Faltantes'] > 0].sort_values('Faltantes', ascending=False)

### Devido a baixa quantidades de faltantes, não foi necessário a exclusão da coluna.

### 4.2 Outros tipos de faltantes
#### Analisando a tabela, verificamos que existe células preenchidas com texto que podem ser considerados 'faltantes'.

In [ ]:
# cria um df com as colunas %NULO e %
nulos = pd.DataFrame({
    # percorre o df original e soma a quantidade de #NULO
    '#NULO': (df_original == '#NULO').sum(),
    # percorre o df original, soma a quantidade de #NULO e calcula a %
    '%': ((df_original == '#NULO').mean() * 100).round(2),
})
# filtra somente as linhas que contém #NULO e depois ordena da maior para o menor
nulos[nulos['#NULO'] > 0].sort_values('#NULO', ascending=False)

In [ ]:
naodivulgavel = pd.DataFrame({
    'NÃO DIVULGÁVEL': (df_original == 'NÃO DIVULGÁVEL').sum(),
    '%': ((df_original == 'NÃO DIVULGÁVEL').mean() * 100).round(2),
})

naodivulgavel[naodivulgavel['NÃO DIVULGÁVEL'] > 0].sort_values('NÃO DIVULGÁVEL', ascending=False)

### 4.3. Substituindo por NaN
Como #NULO e NÃO DIVULGAVEL são strings, é necessário transformar em NaN.

In [ ]:
df_original = df_original.replace('#NULO', np.nan)
df_original = df_original.replace('NÃO DIVULGÁVEL', np.nan)

### 4.4. Contagem de Faltantes após o tratamento

In [ ]:
faltantes_atualizado = pd.DataFrame({
    'Faltantes_atualizado': df_original.isna().sum(),
    '%': (df_original.isna().mean() * 100).round(2),
})

# filta somente as colunas que não possui faltantes
# sort e ascending: ordena pela coluna faltante do maior para o menor
faltantes_atualizado[faltantes_atualizado['Faltantes_atualizado'] > 0].sort_values('Faltantes_atualizado', ascending=False)

Em razão do elevado número de dados faltantes, as colunas DS_EMAIL e NM_SOCIAL_CANDIDATO serão excluídas. Quanto às colunas NM_FEDERACAO, SG_FEDERACAO e DS_COMPOSICAO_FEDERACAO, apesar de também apresentarem uma quantidade alta, é necessário uma análise melhor.

In [ ]:
df_original = df_original.drop(columns=['DS_EMAIL'])
df_original = df_original.drop(columns=['NM_SOCIAL_CANDIDATO'])

## 5. Tratamento de Duplicados
      

In [ ]:
# verifica se existe linhas com todos os dados duplicados
print('Linhas totalmente duplicadas:', df_original.duplicated().sum())

## 6. Tratamento da base auxiliar

Visando atender à exigência de uso do Merge e de três colunas numéricas, utilizou-se a tabela consulta_cand_2022_BRASIL, selecionando os campos NR_IDADE_DATA_POSSE e VR_DESPESA_MAX_CAMPANHA.

**Observação** = VR_DESPESA_MAX_CAMPANHA é o máximo que o candidato pode utilizar. Portanto, não quer dizer que foi utilizado tudo.

### 6.1. Lendo a base auxiliar

In [ ]:
# substitua o caminho abaixo pela pasta onde está salvo a base de dados .csv
# não esquecer que o nome do arquivo e a extensão deve estar inclusos

## pasta_arquivo = "/Users/felipealbanez/Github/CAIXAVERSO_DFF/Projeto_Final_TSE_2022/consulta_cand_complementar_2022_BRASIL.csv"

pasta_arquivo = "../dados/consulta_cand_complementar_2022_BRASIL.csv"

# cria o dataframe lendo o arquivo .csv
auxiliar = pd.read_csv(pasta_arquivo, sep=';', decimal=',',encoding='latin-1')

# seleciona a chave e apenas as duas colunas necessárias
# o SQ_CANDIDATO é número sequencial identificador único do candidato dentro da base do TSE
# funciona como um "id"
auxiliar = auxiliar[['SQ_CANDIDATO','NR_IDADE_DATA_POSSE', 'VR_DESPESA_MAX_CAMPANHA']].copy()


# Converte as colunas para número.
auxiliar['NR_IDADE_DATA_POSSE'] = pd.to_numeric(
    auxiliar['NR_IDADE_DATA_POSSE'],
    errors='coerce'
)
auxiliar['VR_DESPESA_MAX_CAMPANHA'] = pd.to_numeric(
    auxiliar['VR_DESPESA_MAX_CAMPANHA'],
    errors='coerce'
)

auxiliar.head()

### 6.2 Avaliando a base auxiliar

In [ ]:
# identifica idades inválidas
idades_invalidas = auxiliar[(auxiliar['NR_IDADE_DATA_POSSE'] < 0) ]
print('Idades inválidas:', len(idades_invalidas))

# identifica limites de despesas negativos
despesas_invalidas = auxiliar[auxiliar['VR_DESPESA_MAX_CAMPANHA'] < 0]
print('Despesas negativas:', len(despesas_invalidas))

### 6.3. Os valores negativos são subistituídos por NaN

In [ ]:
auxiliar.loc[
    # selecione as linhas com valores negativos
    auxiliar['VR_DESPESA_MAX_CAMPANHA'] < 0,
    'VR_DESPESA_MAX_CAMPANHA'
] = np.nan

# confirma que não existem mais despesas negativas
despesas_invalidas = auxiliar[auxiliar['VR_DESPESA_MAX_CAMPANHA'] < 0]
print('Despesas negativas (depois):', len(despesas_invalidas))

## 7. Junçao das bases com Merge
**SQ_CANDIDATO** é um número sequencial identificador único do candidato dentro da base do TSE.

In [ ]:
# merge com a chave SQ_CANDIDATO
df_merge = df_original.merge(auxiliar, on='SQ_CANDIDATO',how='left')

# dimensoes de antes e depois
print("Shape antes do merge:", df_original.shape)
print("Shape depois do merge:", df_merge.shape)

## 8. Criação de colunas
### Criação de colunas utilizando **np.where**, **pd.cut** e **pd.qcut**.

### 8.1 Coluna Eleito com **np.where**

In [ ]:
# verificando os valores da coluna
print(df_merge['DS_SIT_TOT_TURNO'].value_counts(dropna=False))
print()

#### Tabela de Eleitos

In [ ]:
# criando a coluna ELEITO
df_merge['ELEITO'] = np.where(
    (df_merge['DS_SIT_TOT_TURNO'] == 'ELEITO') |
    (df_merge['DS_SIT_TOT_TURNO'] == 'ELEITO POR QP') |
    (df_merge['DS_SIT_TOT_TURNO'] == 'ELEITO POR MÉDIA'),'SIM','NÃO'
)

df_merge[['SQ_CANDIDATO','ELEITO']].head()

### 8.2 Coluna Faixa Etária com **cut**

In [ ]:
df_merge['FAIXA_ETARIA'] = pd.cut(df_merge['NR_IDADE_DATA_POSSE'], bins=[0, 30, 45, 60, 200],
                              labels=['<30', '30-45', '45-60', '60+'])
print(df_merge['FAIXA_ETARIA'].value_counts().sort_index())

### 8.3 Coluna com **qcut**
Observação: Para resolver o erro foi criada uma coluna com o ranking de despesas, isso auxilia a desempatar valores repetidos e garante que o qcut sempre terá limites únicos.

In [108]:
df_merge['DESPESA_RANK'] = df_merge['VR_DESPESA_MAX_CAMPANHA'].rank(method='first')


In [ ]:
df_merge['QUARTIL_DESPESA'] = pd.qcut(
    df_merge['DESPESA_RANK'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4']
)

print(df_merge['QUARTIL_DESPESA'].value_counts().sort_index())


#ERRO - valores estão como str? = corrigido

#ERRO - Avaliar se der tempo
#ValueError: Bin edges must be unique: Index([1270629.01, 1270629.01, 1270629.01, 3176572.53, 88944030.8], dtype='float64', name='VR_DESPESA_MAX_CAMPANHA').
# You can drop duplicate edges by setting the 'duplicates' kwarg

## 10. Média geral da idade
Uso do NumPy para calculo da idade média geral.

In [ ]:
# calcula a idade média geral usando numpy
idade_media_geral = np.mean(auxiliar['NR_IDADE_DATA_POSSE'])

print('Idade média geral:', round(idade_media_geral, 2), 'anos.')

## 11. Perguntas e Gráficos

### P01) Quais cargos possuem mais candidaturas?

In [ ]:
cargos = df_original.groupby('DS_CARGO')['SQ_CANDIDATO'].count().sort_values(ascending=False)
print("---Quantidade Total por Cargo ---")
print(cargos)

#### P01.1 Gráfico Volume de Candidaturas por Cargo

In [ ]:
cargos.sort_values().plot(kind='barh', figsize=(10, 6), color='steelblue')
plt.title('Volume de Candidaturas por Cargo (Eleições 2022)', fontsize=14)
plt.xlabel('Número de Candidatos', fontsize=12)
plt.ylabel('Cargo Disputado', fontsize=12)
plt.tight_layout()

# Mostra o gráfico na tela
plt.show()


cargo_campeao = cargos.index[0] 
qtd_campeao = cargos.iloc[0]

print(f"📌 LEITURA DOS DADOS:")
print(f"O gráfico evidencia uma disparidade natural na estrutura política brasileira. "
      f"O foco absoluto das candidaturas concentra-se no poder legislativo, liderado pelo cargo de {cargo_campeao}, "
      f"que sozinho acumula {qtd_campeao} registros. Isso acontece porque o número de cadeiras "
      f"disponíveis para deputados é significativamente maior do que para cargos do poder executivo "
      f"(como Governador e Presidente), que aparecem na base do gráfico com os menores volumes.")

### P02) Como as candidaturas se distribuem por gênero?

In [ ]:
generos = df_original.groupby('DS_GENERO')['SQ_CANDIDATO'].count().sort_values(ascending=False)
print("\n--- Quantidade Total por Gênero ---")
print(generos)

#### P02.1 Gráfico de Distribuição de candidaturas por Gênero

In [ ]:
generos.plot(kind='bar', figsize=(8, 5), color=['#2ca02c', '#9467bd', '#7f7f7f'])
plt.title('Distribuição de Candidaturas por Gênero', fontsize=14)
plt.ylabel('Número de Candidatos', fontsize=12)
plt.xticks(rotation=0) # Mantém os nomes retos no eixo X
plt.tight_layout()

# Mostra o gráfico na tela
plt.show()

genero_maioria = generos.index[0]
percentual = (generos.iloc[0] / generos.sum()) * 100

print(f"📌 LEITURA DOS DADOS:")
print(f"A visualização demonstra que a distribuição de gênero nas eleições ainda é bastante desigual. "
      f"O gênero {genero_maioria} representa a maioria absoluta, compondo aproximadamente {percentual:.1f}% "
      f"de todas as candidaturas registradas. Esse cenário levanta discussões importantes sobre a "
      f"representatividade na política e mostra que, apesar das cotas partidárias existentes, "
      f"a base de candidatos ainda possui uma forte concentração em um único grupo demográfico.")

#### P02.2 Distribuição de Gênero por cargo Político

In [ ]:
comparacao = pd.pivot_table(
    df_original, 
    index='DS_CARGO',       
    columns='DS_GENERO',   
    values='SQ_CANDIDATO',  
    aggfunc='count',       
    fill_value=0
)

print("\n--- Tabela Dinâmica: Cargo x Gênero ---")
display(comparacao)

### P03) Como as candidaturas se distribuem por raça/cor?

In [ ]:
# conta as candidaturas por raça/cor.
por_raca = df_original['DS_COR_RACA'].value_counts()

# calcula o percentual.
percentual_raca = (por_raca / por_raca.sum() * 100).round(2)

# mostra quantidade e percentual.
display(pd.DataFrame({'quantidade': por_raca,'percentual': percentual_raca}))

#### P03.1 Gráfico de candidatura por raça/cor

In [ ]:
# gráfico
por_raca.sort_values().plot(kind='barh', figsize=(8, 5))
plt.title('Candidaturas por raça/cor')
plt.xlabel('Quantidade')
plt.ylabel('Raça/cor')
plt.tight_layout()
plt.show()

#### P03.2 Leitura do gráfico

In [ ]:
print('Conclusão:', por_raca.index[0], 'é a categoria mais frequente, com', percentual_raca.iloc[0], '%.')

### P04) Relação entre Limite de Despesas e Resultado Eleitoral

Nesta seção investigamos se existe uma relação descritiva entre a capacidade financeira dos candidatos  
(limite máximo de despesas permitido) e o resultado eleitoral.



In [ ]:
print("Diagnóstico inicial da variável VR_DESPESA_MAX_CAMPANHA:")
print(df_merge['VR_DESPESA_MAX_CAMPANHA'].describe().round(2))
print("\nValores faltantes:", df_merge['VR_DESPESA_MAX_CAMPANHA'].isna().sum())

Q1 = df_merge['VR_DESPESA_MAX_CAMPANHA'].quantile(0.25)
Q3 = df_merge['VR_DESPESA_MAX_CAMPANHA'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers = df_merge[
    (df_merge['VR_DESPESA_MAX_CAMPANHA'] < limite_inferior) |
    (df_merge['VR_DESPESA_MAX_CAMPANHA'] > limite_superior)
]

print("\nOutliers identificados:", len(outliers))


#### P04.1 Tratamento dos Outliers

Valores extremos de limite de despesas podem distorcer a análise, especialmente quando queremos comparar  
grupos de candidatos. Por isso, aplicamos o método IQR para identificar outliers e utilizamos `clip()`  
para limitar os valores ao intervalo considerado razoável.



In [ ]:
df_merge['DESPESA_TRATADA'] = df_merge['VR_DESPESA_MAX_CAMPANHA'].clip(
    lower=limite_inferior,
    upper=limite_superior
)

print(df_merge['DESPESA_TRATADA'].describe().round(2))


#### P04.2 Tabela Dinâmica: Quartil de Despesas × Resultado Eleitoral

A tabela abaixo mostra quantos candidatos foram eleitos ou não em cada quartil de despesas.


In [ ]:
pivot = df_merge.pivot_table(
    index='QUARTIL_DESPESA',
    columns='ELEITO',
    values='SQ_CANDIDATO',
    aggfunc='count',
    fill_value=0
)

display(pivot)


#### P04.3 Taxa de Eleição por Quartil

Aqui calculamos a taxa percentual de candidatos eleitos em cada quartil.


In [ ]:
taxa = (pivot['SIM'] / pivot.sum(axis=1) * 100).round(2)
print(taxa)


#### P04.4 Gráfico: Taxa de Eleição por Quartil de Despesas

O gráfico abaixo mostra como a taxa de eleição varia conforme o limite de despesas.


In [ ]:
plt.figure(figsize=(8,5))
taxa.plot(kind='bar', color='steelblue')
plt.title('Taxa de eleição por quartil de despesas tratadas')
plt.ylabel('Taxa de eleição (%)')
plt.xlabel('Quartil de despesas')
plt.tight_layout()
plt.show()


#### P04.5 Leitura dos Dados e conclusão

A análise revela uma relação clara entre o limite de despesas e o resultado eleitoral.  
O quartil com maior capacidade financeira apresenta a maior taxa de eleição.

Isso indica que candidatos inseridos nos grupos com maior limite de despesas **tendem** a obter melhores resultados.  
Essa é uma **associação descritiva**, não causal — não podemos afirmar que gastar mais causa a vitória.

